# 01 — Raw inverted index
Find the repository by walking upward, exactly as notebooks 08 and 09 do. Data lives under `data-pipeline/data`. Import the reusable class from this notebook's folder. No text is cleaned or normalized.

In [1]:
import json
import sys
import unicodedata
from pathlib import Path


def find_project_root(start: Path) -> Path:
    """Find the nearest ancestor containing the data-pipeline directory."""
    for directory in (start, *start.parents):
        if (directory / 'data-pipeline').is_dir():
            return directory
    raise RuntimeError('Run this notebook inside the legal-rag project')


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / 'data-pipeline' / 'data'
NOTEBOOK_DIR = PROJECT_ROOT / 'data-pipeline' / 'notebook_bm25'
INPUT_PATH = DATA_DIR / 'processed' / 'articles.jsonl'
OUTPUT_DIR = DATA_DIR / 'bm25' / 'index_raw'
sys.path.insert(0, str(NOTEBOOK_DIR))
from bm25_index import InvertedIndex, canonical_id

print('Project root:', PROJECT_ROOT)
print('Input:', INPUT_PATH)
print('Output:', OUTPUT_DIR)

Project root: /home/panha/ir/project/legal-rag
Input: /home/panha/ir/project/legal-rag/data-pipeline/data/processed/articles.jsonl
Output: /home/panha/ir/project/legal-rag/data-pipeline/data/bm25/index_raw


Load each nonblank JSON line as one article record. Print the record count, available fields, and one complete sample so the input can be checked.

In [2]:
with INPUT_PATH.open(encoding='utf-8') as source:
    articles = [json.loads(line) for line in source if line.strip()]
print('Loaded records:', len(articles))
print('Fields:', sorted(articles[0]) if articles else [])
print(json.dumps(articles[0] if articles else {}, ensure_ascii=False, indent=2))

Loaded records: 20268
Fields: ['article_id', 'article_number', 'document_id', 'document_number', 'document_title', 'document_type', 'id', 'pdf_url', 'source_url', 'text']
{
  "id": "1-1",
  "article_id": "1-1",
  "document_id": "1",
  "article_number": 1,
  "text": "មាត្រា ៨ មាត្រា ៤០ មាត្រា ៤១ មាត្រា ៤៧ មាត្រា ៤៨ មាត្រា ៧៥ មាត្រា ៧៧ មាត្រា ៧៨ និងមាត្រា ៨០ នៃច្បាប់ស្តីពីចរាចរណ៍ផ្លូវគោក ដែលត្រូវប្រកាសឱ្យប្រើដោយព្រះរាជក្រមលេខ នស/រកម/០១១៥/០០១ ចុះថ្ងៃទី ៩ ខែ មករា ឆ្នាំ២០១៥ ត្រូវបានធ្វើវិសោធនកម្មដូចតទៅ ៖",
  "document_title": "1.txt",
  "document_number": "នស/រកម/០១១៧/០០២",
  "document_type": "unknown",
  "source_url": "unknown",
  "pdf_url": "unknown"
}


Build the index using only `text.split()`, which separates text at whitespace. A posting list maps article IDs to term frequency (the number of occurrences in that article). Document frequency counts articles containing a token.

As in dense notebooks 08 and 09, use `article_id`, falling back to `document_id-article_number`. For repeated IDs, keep the longest raw text; ties keep the first record. This avoids counting the same canonical article twice.

In [3]:
index = InvertedIndex.build(articles)
print('Unique documents:', index.total_documents)
print('Repeated-ID records collapsed:', len(articles) - index.total_documents)
print('Total token occurrences:', sum(index.document_lengths.values()))
print('Average document length:', round(index.average_document_length, 2))
print('Sample document lengths:', list(index.document_lengths.items())[:5])

Unique documents: 17540
Repeated-ID records collapsed: 2728
Total token occurrences: 580074
Average document length: 33.07
Sample document lengths: [('1-1', 28), ('1-8', 120), ('1-40', 97), ('1-41', 154), ('1-47', 36)]


Inspect three common tokens and the top 20 tokens, ranked by total occurrences across all documents. Limit posting previews to five entries for readability; the complete lists remain in `index.postings`. Also count tokens found in only one document, even if repeated there.

In [4]:
TOP_TOKEN_COUNT = 20  # Number of vocabulary entries to inspect.
POSTING_PREVIEW_COUNT = 5  # Keep each posting preview short.
POSTING_TOKEN_COUNT = 3  # Inspect the three most common tokens.
frequencies = index.collection_frequencies
document_frequencies = index.document_frequencies
ranked_tokens = sorted(frequencies, key=lambda token: (-frequencies[token], token))
for token in ranked_tokens[:POSTING_TOKEN_COUNT]:
    print(repr(token), 'document frequency:', document_frequencies[token],
          'posting preview:', list(index.postings[token].items())[:POSTING_PREVIEW_COUNT])
print('Top tokens: token | total occurrences | document frequency')
for token in ranked_tokens[:TOP_TOKEN_COUNT]:
    print(repr(token), frequencies[token], document_frequencies[token])
print('Tokens present in exactly one document:', sum(df == 1 for df in document_frequencies.values()))

'៖' document frequency: 6030 posting preview: [('1-1', 1), ('1-8', 17), ('1-40', 10), ('1-41', 16), ('1-47', 6)]
'។' document frequency: 7218 posting preview: [('1004-1', 1), ('1004-2', 1), ('1004-3', 3), ('1004-4', 2), ('1004-5', 1)]
'the' document frequency: 1120 posting preview: [('11168-37', 1), ('11390-1', 9), ('11390-2', 20), ('11390-3', 11), ('11390-4', 14)]
Top tokens: token | total occurrences | document frequency
'៖' 17285 6030
'។' 17249 7218
'the' 9905 1120
'of' 8608 1173
'-' 7882 1653
'ឬ' 7501 2692
'៕' 5197 2130
'to' 4954 1091
'និង' 4749 2274
'ឆ្នាំ' 4460 2324
'|' 3766 69
'and' 3678 885
'or' 3615 845
'in' 2863 836
'a' 2670 811
':' 2632 1584
'shall' 2343 1003
'១' 2336 1484
'ខែ' 2154 1616
'be' 1903 985
Tokens present in exactly one document: 161752


Boolean AND search returns only articles containing every exact query token. The class intersects posting lists (keeps the IDs shared by all lists). Try two Khmer queries without cleaning their text; whitespace boundaries must match the raw articles.

In [5]:
EXAMPLE_QUERIES = ['មាត្រា និង', 'ច្បាប់ នៃ']
SEARCH_PREVIEW_COUNT = 5  # Show only the first five matching article IDs.
for query in EXAMPLE_QUERIES:
    matches = index.boolean_and(query.split())
    print(repr(query), 'matches:', len(matches), 'sample IDs:', matches[:SEARCH_PREVIEW_COUNT])
    assert all(all(doc_id in index.postings[token] for token in query.split()) for doc_id in matches)
print('Empty query:', index.boolean_and([]))
assert index.boolean_and(['__absent_raw_token__']) == []

'មាត្រា និង' matches: 166 sample IDs: ['1004-61', '1004-66', '1053-135', '1053-136', '1053-54']
'ច្បាប់ នៃ' matches: 15 sample IDs: ['1231-3', '1507-158', '2-20', '396-21', '4118-10']
Empty query: []


Measure raw vocabulary noise. Vocabulary means the set of distinct tokens. The one-time share is the number of distinct tokens with exactly one total occurrence divided by vocabulary size; this differs from appearing in one document. Average token length counts Unicode code points per distinct token, not visual Khmer characters.

Show the longest Khmer-containing tokens as possible joined phrases, and tokens with punctuation or digits attached to letters. These are inspection hints, not automatic judgments of linguistic correctness. Nothing is removed.

In [6]:
EXAMPLE_COUNT = 5  # Show five examples in each noise category.
vocabulary_size = len(frequencies)
one_time_count = sum(count == 1 for count in frequencies.values())
one_time_share = one_time_count / vocabulary_size if vocabulary_size else 0.0
average_token_length = sum(map(len, frequencies)) / vocabulary_size if vocabulary_size else 0.0
print('Vocabulary size:', vocabulary_size)
print('Tokens occurring exactly once:', one_time_count)
print(f'One-time vocabulary share: {one_time_share:.2%}')
print(f'Average distinct-token length (Unicode code points): {average_token_length:.2f}')
khmer_tokens = [token for token in frequencies if any('KHMER' in unicodedata.name(char, '') for char in token)]
longest_tokens = sorted(khmer_tokens, key=lambda token: (-len(token), token))
print('Longest Khmer-containing tokens (possible joined phrases):', longest_tokens[:EXAMPLE_COUNT])
for label, category_prefix in [('Punctuation attached to letters', 'P'), ('Numbers attached to letters', 'N')]:
    examples = [token for token in longest_tokens
                if any(unicodedata.category(char).startswith('L') for char in token)
                and any(unicodedata.category(char).startswith(category_prefix) for char in token)]
    print(label + ':', examples[:EXAMPLE_COUNT])

Vocabulary size: 193323
Tokens occurring exactly once: 157958
One-time vocabulary share: 81.71%
Average distinct-token length (Unicode code points): 31.55


Longest Khmer-containing tokens (possible joined phrases): ['នៃអនុសញ្ញានេះភាគីរដ្ឋនីមួយៗត្រូវអនុវត្តបណ្ដាវិធានការដូចដែលចាំបាច់ដើម្បីបង្កើតយុត្តាធិការរបស់ខ្លួនលើបទល្មើសទាំងឡាយដែលបានចែងស្របតាមអនុសញ្ញានេះនៅពេលដែលជនដែលត្រូវបានចោទប្រកាន់មានវត្តមាននៅក្នុងដែនដីរបស់ខ្លួនហើយមិនបានធ្វើការបណ្ដេញចេញឬចាត់ចែងលើកទីកតាំងដែលជាជាតិចម្បងនៃសញ្ជាតិនោះ', 'ប្រធានគណៈកម្មាធិការជាតិរៀបចំការបោះឆ្នោតគឺជាអាណាប័កដើមនៃគណនីពិសេសរបស់ធនាគារជាតិសំរាប់ការបោះឆ្នោតជ្រើសតាំងតំណាងរាស្ត្រនិងត្រូវមានអាណាប័កតាមការប្រគល់សិទ្ធិម្នាក់ឬច្រើននាក់ជាជំនួយការដែលតែងតាំងឡើងនិងស្ថិតក្រោមការទទួលខុសត្រូវរបស់អាណាប័កដើមថ្លៃផ្ទាល់', 'ការសម្រេចឱ្យប្រើវិធីប្រគល់ដោយមានការឃ្លាំមើលឬប្រើវិធីលក់ដោយខុសច្បាប់ត្រូវធ្វើដោយអាជ្ញាធរដែលចាត់តាំងដោយប្រកាសរបស់រដ្ឋមន្ត្រីក្រសួងយុត្តិធម៌និងដោយរដ្ឋមន្ត្រីប្រព្រឹត្តដែលរដ្ឋមន្ត្រីក្រសួងយុត្តិធម៌ចាត់តាំងសហការលើករណីនីមួយៗហើយលើសពីជាច្រើយធ្វើ', 'ក្នុងផ្នែកនេះមិនតម្រូវឱ្យសមាជិកអនុវត្តបទប្បញ្ញត្តិទាក់ទងនឹងម៉ាកសម្គាល់ភូមិសាស្ត្រទំនិញរបស់សមាជិកណាមួយផ្សេងទៀតចំពោះផលិតផលស្រាទំពាំងបាយជូរដែលពាក់ព័ន្ធនឹងម៉ាកកែកូដដែលឈ្មោះទូទៅនៃប្

Punctuation attached to letters: ['បុគ្គលឬអង្គភាពក្រៅពីបុគ្គលរាយការណ៍ដែលបានធ្វើរបាយការណ៍ស្របតាមមាត្រានេះត្រូវផ្តល់ព័ត៌មានផ្សេងទៀតដែលទាក់ទងនឹងរបាយការណ៍នេះឱ្យបានឆាប់រហ័សតាមសំណើរបស់ក្រសួងយុត្តិធម៌ឬអង្គភាពរាយការណ៍ហិរញ្ញវត្ថុកម្ពុជា។', 'អចលនវត្ថុជាដីនិងជាសំណង់ដែលមាននៅក្នុងនិងក្រៅបរិវេណវត្តអារាមនៃព្រះពុទ្ធសាសនាជាបេតិកភណ្ឌប្រើប្រាស់ជាកិច្ចការសំរាប់ព្រះពុទ្ធសាសនានិងដាក់ឱ្យពុទ្ធសាសនិកប្រើប្រាស់ក្រោមការថែទាំរបស់គណៈកម្មការវត្ត។', 'បុគ្គលនៃវិស័យសាធារណៈជាស្រ្តីមានសិទ្ធិទទួលបានប្រាក់បំណាច់ពេញលេញក្នុងរយៈពេលសម្រាកមាតុភាពក្នុងរយៈពេលសម្រាកព្យាបាលថែទាំសុខភាពមុននិងក្រោយពេលសម្រាលដូចមានកំណត់ក្នុងលិខិតបទដ្ឋានគតិយុត្តជាធរមាន។', 'អាចចេញលិខិតបទដ្ឋានគតិយុត្តកំណត់លក្ខខណ្ឌលទ្ធកម្មដែលអនុញ្ញាតឱ្យស្ថាប័នអនុវត្តលទ្ធកម្មមានសិទ្ធិកែសម្រួលផែនការលទ្ធកម្មដោយមិនចាំបាច់សុំការពិនិត្យនិងសម្រេចពីក្រសួងសេដ្ឋកិច្ចនិងហិរញ្ញវត្ថុ។', 'ប្រទេសផ្សេងទៀតនៃសហភាពមិនតម្រូវឱ្យផ្តល់ចំពោះស្នាដៃដែលបានដាក់ឱ្យស្ថិតនៅក្រោមបច្ច័យនៃកិច្ចការពារប្រសើរជាងកិច្ចការពារដែលបានផ្តល់ឱ្យស្នាដៃទាំងនោះនៅក្នុងប្រទេសនៃកាលបោះពុម្ពផ្សាយលើកដំបូង។']


Numbers attached to letters: ['តុលាការត្រូវចេញសេចក្តីសម្រេចដាក់ឈ្មោះបុគ្គលឬអង្គភាពក្នុងបញ្ជីនិងបង្កកទ្រព្យសម្បត្តិរបស់គេក្នុងករណីមានមូលដ្ឋានសមហេតុផលនិងជឿជាក់ថាបុគ្គលឬអង្គភាពនោះត្រូវនឹងលក្ខខណ្ឌដូចមានចែងក្នុងតារាងណាមួយនៃសេចក្តីសម្រេចចិត្តលេខ២២៧០', 'ប្រាក់បំណាច់ប្រចាំថ្ងៃសម្រាប់ការសម្រាកព្យាបាលជំងឺនៅតាមមូលដ្ឋានសុខាភិបាលត្រូវកំណត់ស្មើនឹង៧០%(ចិតសិបភាគរយ)នៃប្រាក់ឈ្នួលមធ្យមប្រចាំថ្ងៃដែលបានបង់ភាគទានសម្រាប់រយៈពេលមិនកន្លងលើស១(មួយ)ខែ', 'ខេត្តសាម៉ីលើសំណើសុំចុះបញ្ជីសហគមន៍កសិកម្មដោយធ្វើលិខិតតវ៉ាទៅនាយកដ្ឋានអភិវឌ្ឍន៍សហគមន៍កសិកម្មក្នុងរយៈពេល៣០(សាមសិប)ថ្ងៃយ៉ាងយូរគិតចាប់ពីថ្ងៃទទួលបានសេចក្តីបដិសេធរបស់មន្ទីរកសិកម្មរាជធានី', 'ជាបុគ្គលទាំងឡាយដែលស្ថិតនៅក្រោមបទប្បញ្ញត្តិនៃច្បាប់ស្តីពីការងារដោយរួមបញ្ចូលទាំងបុគ្គលិកធ្វើការតាមផ្លូវអាកាសនិងនាវាសមុទ្រព្រមទាំងអ្នកបម្រើក្នុងផ្ទះដែលមានអាយុយ៉ាងតិច៦០(ហាសិប)ឆ្នាំ', 'ត្រូវមកដាក់ពាក្យស្នើសុំអាជ្ញាប័ណ្ណក្នុងរយៈពេល៣(បី)ខែបន្ទាប់ពីទទួលបានលិខិតជូនដំណឹងពីក្រសួងទទួលបន្ទុកវិស័យទឹកស្អាតឬអង្គភាពដែលទទួលបានប្រតិភូកម្មពីក្រសួងទទួលបន្ទុកវិស័យទឹកស្អាត។']


Save the raw index as UTF-8 JSON in the generated BM25 data folder. Reload it and compare every posting, document length, and corpus statistic. The saved JSON also includes document frequencies, document count, and average document length.

In [7]:
index.save(OUTPUT_DIR)
reloaded = InvertedIndex.load(OUTPUT_DIR)
assert reloaded.postings == index.postings
assert reloaded.document_lengths == index.document_lengths
assert reloaded.document_frequencies == index.document_frequencies
assert reloaded.total_documents == index.total_documents
assert reloaded.average_document_length == index.average_document_length
for query in EXAMPLE_QUERIES:
    assert reloaded.boolean_and(query.split()) == index.boolean_and(query.split())
print('Saved:', OUTPUT_DIR / 'index.json')
print('Saved bytes:', (OUTPUT_DIR / 'index.json').stat().st_size)
print('Reload verified: all postings, lengths, statistics, and example searches match.')

Saved: /home/panha/ir/project/legal-rag/data-pipeline/data/bm25/index_raw/index.json
Saved bytes: 44685435
Reload verified: all postings, lengths, statistics, and example searches match.
